### ЗАДАЧА: Панель SLA-ребейтов по доставке по паттерну `MVC`

Команда logistics finance разбирает кейсы по SLA-ребейтам: если доставка опоздала,
клиенту или продавцу может быть положена компенсация, а к логистическому партнеру — применен штраф.
Нужно реализовать внутреннюю консольную панель по паттерну `MVC`.

Слои:
- `Model` хранит кейсы и бизнес-правила;
- `View` отвечает только за отображение;
- `Controller` принимает действия и связывает `Model` и `View`.

## Что должно храниться в кейсе

Для каждого кейса нужно хранить:
- `case_id` — идентификатор кейса;
- `shipment_id` — идентификатор отправления;
- `courier` — служба доставки;
- `promised_days` — обещанный срок доставки;
- `actual_days` — фактический срок доставки;
- `order_value` — стоимость заказа;
- `shipping_fee` — стоимость доставки;
- `penalty_rate` — ставка компенсации за каждый день просрочки;
- `delay_days` — число дней просрочки;
- `requested_rebate` — расчетная сумма ребейта;
- `approved_rebate` — согласованная сумма ребейта;
- `courier_penalty` — штраф, который будет предъявлен логистическому партнеру;
- `status` — статус кейса;
- `coordinator` — сотрудник, который ведет кейс;
- `customer_contacted` — связывались ли с клиентом;
- `decision` — финальное решение.

## Формулы

При создании кейса и после изменения одобренной суммы нужно считать:
- `delay_days = max(actual_days - promised_days, 0)`
- `requested_rebate = min(order_value * penalty_rate * delay_days, shipping_fee + order_value * 0.2)`
- `approved_rebate` при создании равно `0.0`
- `courier_penalty = approved_rebate * 0.7`
- все денежные значения округляются до 2 знаков.

## Статусы

- `new`
- `investigating`
- `customer_contacted`
- `ready_for_approval`
- `approved`
- `rejected`
- `escalated`

## Бизнес-правила

- нельзя создать кейс с уже существующим `case_id`;
- нельзя назначить `coordinator` несуществующему кейсу;
- финальные кейсы (`approved`, `rejected`, `escalated`) нельзя менять дальше;
- начать расследование можно только из `new` и только если назначен `coordinator`;
- связаться с клиентом можно только из `investigating`;
- при контакте с клиентом поле `customer_contacted` должно стать `True`, а статус — `customer_contacted`;
- установить `approved_rebate` можно только из `investigating` или `customer_contacted`;
- `approved_rebate` не может быть меньше `0`;
- `approved_rebate` не может быть больше `requested_rebate`;
- после изменения `approved_rebate` нужно пересчитать `courier_penalty`;
- перевод в `ready_for_approval` возможен только из `investigating` или `customer_contacted`;
- перевод в `ready_for_approval` возможен только если `approved_rebate > 0`;
- завершить кейс как `approved` можно только из `ready_for_approval`;
- завершить кейс как `rejected` можно только из `ready_for_approval`, если `approved_rebate == 0`;
- `escalated` можно сделать только из `investigating`, `customer_contacted` или `ready_for_approval`;
- при финальном статусе нужно записывать `decision`.

## Что должен уметь `Model`

Нужно самостоятельно спроектировать модель, но она должна уметь минимум:
- создавать кейс;
- назначать координатора;
- начинать расследование;
- отмечать контакт с клиентом;
- устанавливать `approved_rebate`;
- переводить кейс в `ready_for_approval`;
- завершать кейс как `approved`;
- завершать кейс как `rejected`;
- эскалировать кейс;
- возвращать список кейсов;
- возвращать summary.

## Что должен уметь `View`

Нужно реализовать вывод:
- списка кейсов;
- summary;
- успешных сообщений;
- ошибок.

## Формат строки кейса

Каждый кейс можно вывести строкой такого вида:

`case_id | shipment_id | courier | promised_days | actual_days | delay_days | order_value | shipping_fee | requested_rebate | approved_rebate | courier_penalty | status | coordinator | customer_contacted | decision`

## Что должно быть в summary

Нужно вернуть словарь, в котором есть:
- количество кейсов по статусам;
- `total_requested_rebate` — общая расчетная сумма ребейтов;
- `total_approved_rebate` — общая согласованная сумма;
- `total_courier_penalty` — общая сумма штрафов логистическому партнеру;
- `delayed_shipments` — количество кейсов, где `delay_days > 0`;
- `contacted_cases` — количество кейсов, где `customer_contacted == True`.

## Что нужно сделать в конце

1. Создать модель, view и controller.
2. Загрузить `initial_cases`.
3. Обработать все действия из `actions`.
4. В конце вывести список кейсов и summary.

In [7]:

from dataclasses import dataclass

initial_cases = [
    ("DR-100", "SHP-9901", "FastBox", 2, 5, 4200.0, 300.0, 0.03),
    ("DR-101", "SHP-9902", "QuickShip", 3, 3, 1800.0, 220.0, 0.02),
]

actions = [
    ("show",),
    ("investigate", "DR-100"),
    ("assign", "DR-100", "Olga"),
    ("investigate", "DR-100"),
    ("contact", "DR-100"),
    ("set_rebate", "DR-100", 180.0),
    ("ready", "DR-100"),
    ("approve", "DR-100", "rebate_sent_to_customer"),
    ("create", "DR-102", "SHP-9903", "CityRun", 1, 4, 2600.0, 180.0, 0.04),
    ("assign", "DR-102", "Max"),
    ("investigate", "DR-102"),
    ("set_rebate", "DR-102", 150.0),
    ("ready", "DR-102"),
    ("escalate", "DR-102", "courier_disputes_delay_window"),
    ("create", "DR-103", "SHP-9904", "ParcelWay", 2, 6, 5200.0, 450.0, 0.03),
    ("assign", "DR-103", "Ina"),
    ("investigate", "DR-103"),
    ("set_rebate", "DR-103", 0.0),
    ("ready", "DR-103"),
    ("reject", "DR-103", "no_customer_refund_required"),
    ("show",),
]


@dataclass
class SLACase:
    case_id: str
    shipment_id: str
    courier: str
    promised_days: int
    actual_days: int
    order_value: float
    shipping_fee: float
    penalty_rate: float
    delay_days: int = 0
    requested_rebate: float = 0.0
    approved_rebate: float = 0.0
    courier_penalty: float = 0.0
    status: str = "new"
    coordinator: str | None = None
    customer_contacted: bool = False
    decision: str | None = None
    
    def calculate_values(self) -> None:
        self.delay_days = max(self.actual_days - self.promised_days, 0)
        max_rebate = self.shipping_fee + self.order_value * 0.2
        calculated_rebate = self.order_value * self.penalty_rate * self.delay_days
        self.requested_rebate = round(min(calculated_rebate, max_rebate), 2)
        self.courier_penalty = round(self.approved_rebate * 0.7, 2)
        
class SLAModel:
    def __init__(self):
        self.cases = {}
        
    def create_case(self, case_id: str, shipment_id: str, courier: str, promised_days: int, actual_days: int, order_value: float, shipping_fee: float, penalty_rate: float) -> None:  
        if case_id in self.cases:
            raise ValueError("Case with this ID already exists")
        case = SLACase(case_id, shipment_id, courier, promised_days, actual_days, order_value, shipping_fee, penalty_rate)  
        case.calculate_values()
        self.cases[case_id] = case

        
    def assign_coordinator(self, case_id: str, coordinator: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        if case.status in ("approved", "rejected", "escalated"):
            raise ValueError("Final cases cannot be modified")
        case.coordinator = coordinator
        
    def start_investigation(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        if case.status != "new":
            raise ValueError("Investigation can only be started from new status")
        if not case.coordinator:
            raise ValueError("Coordinator is required")
        case.status = "investigating"
        
    def contact_customer(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        if case.status != "investigating":
            raise ValueError("Contact with customer can only be made from investigating status")
        case.customer_contacted = True
        case.status = "customer_contacted"
        
    def set_approved_rebate(self, case_id: str, rebate_amount: float) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        if case.status not in ("investigating", "customer_contacted"):
            raise ValueError("Approved rebate can only be set from investigating or customer_contacted status")
        if rebate_amount < 0:
            raise ValueError("Approved rebate cannot be negative")
        if rebate_amount > case.requested_rebate:
            raise ValueError("Approved rebate cannot exceed requested rebate")
        case.approved_rebate = rebate_amount
        case.courier_penalty = round(rebate_amount * 0.7, 2)
        
    def mark_ready_for_approval(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        if case.status not in ("investigating", "customer_contacted"):
            raise ValueError("Ready for approval can only be marked from investigating or customer_contacted status")
        if case.approved_rebate <= 0:
            raise ValueError("Approved rebate must be greater than 0 to mark ready for approval")
        case.status = "ready_for_approval"
        
    def approve_case(self, case_id: str, decision: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        if case.status != "ready_for_approval":
            raise ValueError("Case can only be approved from ready_for_approval status")
        case.status = "approved"
        case.decision = decision
        
    def reject_case(self, case_id: str, decision: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        if case.status != "ready_for_approval":
            raise ValueError("Case can only be rejected from ready_for_approval status")
        if case.approved_rebate != 0:
            raise ValueError("Rebate must be 0 to reject case")
        case.status = "rejected"
        case.decision = decision
        
    def escalate_case(self, case_id: str, decision: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        if case.status in ("approved", "rejected", "escalated"):
            raise ValueError("Final cases cannot be modified")
        if case.status not in ("new", "investigating", "customer_contacted", "ready_for_approval"):
            raise ValueError("Case can only be escalated from valid intermediate statuses")
        case.status = "escalated"
        case.decision = decision
    
    def list_cases(self) -> list[str]:
        rows = []
        for case in self.cases.values():
            coordinator = case.coordinator if case.coordinator is not None else "None"
            decision = case.decision if case.decision is not None else "None"
            rows.append(
                f"{case.case_id} | {case.shipment_id} | {case.courier} | {case.promised_days} | "
                f"{case.actual_days} | {case.delay_days} | {case.order_value} | {case.shipping_fee} | "
                f"{case.requested_rebate} | {case.approved_rebate} | {case.courier_penalty} | "
                f"{case.status} | {coordinator} | {case.customer_contacted} | {decision}"
            )
        return rows
    
    def summary(self) -> dict[str, float | int]:
        result = {
            "total_requested_rebate": 0.0,
            "total_approved_rebate": 0.0,
            "total_courier_penalty": 0.0,
            "delayed_shipments": 0,
            "contacted_cases": 0
        }
        
        all_statuses = ["new", "investigating", "customer_contacted",
                      "ready_for_approval", "approved", "rejected", "escalated"]
        for status in all_statuses:
            result[status] = 0
            
        for case in self.cases.values():
            if case.status in result:
                result[case.status] += 1
            result["total_requested_rebate"] += case.requested_rebate
            result["total_approved_rebate"] += case.approved_rebate
            result["total_courier_penalty"] += case.courier_penalty
            if case.delay_days > 0:
                result["delayed_shipments"] += 1
            if case.customer_contacted:
                result["contacted_cases"] += 1
        return result
                       
class SLAView:
    @staticmethod
    def render_cases(rows: list[str]) -> None:
        print("SLA Rebate Cases:")
        for row in rows:
            print(row)
            
    @staticmethod
    def render_summary(summary: dict[str, float | int]) -> None:
        print("\nSummary:")
        for key, value in summary.items():
            print(f"  {key}: {value}")
            
    @staticmethod
    def render_success(message: str) -> None:
        print(f"SUCCESS: {message}")
        
    @staticmethod
    def render_error(message: str) -> None:
        print(f"ERROR: {message}")
        
class SLAController:
    def __init__(self, model: SLAModel, view: SLAView):
        self.model = model
        self.view = view
        
    def create_case(self, case_id: str, shipment_id: str, courier: str, promised_days: int,
                   actual_days: int, order_value: float, shipping_fee: float, penalty_rate: float) -> None:
        try:
            self.model.create_case(case_id, shipment_id, courier, promised_days, actual_days,
                             order_value, shipping_fee, penalty_rate)
            self.view.render_success(f"Case {case_id} created")
        except ValueError as error:
            self.view.render_error(str(error))
            
    def assign_coordinator(self, case_id: str, coordinator: str) -> None:
        try:
            self.model.assign_coordinator(case_id, coordinator)
            self.view.render_success(f"Coordinator assigned to {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))
            
    def start_investigation(self, case_id: str) -> None:
        try:
            self.model.start_investigation(case_id)
            self.view.render_success(f"Investigation started for {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))
            
    def contact_customer(self, case_id: str) -> None:
        try:
            self.model.contact_customer(case_id)
            self.view.render_success(f"Customer contacted for {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))
            
    def set_approved_rebate(self, case_id: str, rebate_amount: float) -> None:
        try:
            self.model.set_approved_rebate(case_id, rebate_amount)
            self.view.render_success(f"Approved rebate set for {case_id}: {rebate_amount}")
        except ValueError as error:
            self.view.render_error(str(error))
            
    def mark_ready_for_approval(self, case_id: str) -> None:
        try:
            self.model.mark_ready_for_approval(case_id)
            self.view.render_success(f"Case {case_id} marked ready for approval")
        except ValueError as error:
            self.view.render_error(str(error))
            
    def approve_case(self, case_id: str, decision: str) -> None:
        try:
            self.model.approve_case(case_id, decision)
            self.view.render_success(f"Case {case_id} approved")
        except ValueError as error:
            self.view.render_error(str(error))
            
    def reject_case(self, case_id: str, decision: str) -> None:
        try:
            self.model.reject_case(case_id, decision)
            self.view.render_success(f"Case {case_id} rejected")
        except ValueError as error:
            self.view.render_error(str(error))
            
    def escalate_case(self, case_id: str, decision: str) -> None:
        try:
            self.model.escalate_case(case_id, decision)
            self.view.render_success(f"Case {case_id} escalated")
        except ValueError as error:
            self.view.render_error(str(error))

    def show_cases(self) -> None:
        self.view.render_cases(self.model.list_cases())
        self.view.render_summary(self.model.summary())
        
model = SLAModel()
view = SLAView()
controller = SLAController(model, view)

for case_data in initial_cases:
    controller.create_case(*case_data)
    
for action in actions:
    if action[0] == "show":
        controller.show_cases()
    elif action[0] == "create":
        _, case_id, shipment_id, courier, promised_days, actual_days, order_value, shipping_fee, penalty_rate = action
        controller.create_case(case_id, shipment_id, courier, promised_days, actual_days,
                    order_value, shipping_fee, penalty_rate)
    elif action[0] == "assign":
        _, case_id, coordinator = action
        controller.assign_coordinator(case_id, coordinator)
    elif action[0] == "investigate":
        _, case_id = action
        controller.start_investigation(case_id)
    elif action[0] == "contact":
        _, case_id = action
        controller.contact_customer(case_id)
    elif action[0] == "set_rebate":
        _, case_id, rebate_amount = action
        controller.set_approved_rebate(case_id, rebate_amount)
    elif action[0] == "ready":
        _, case_id = action
        controller.mark_ready_for_approval(case_id)
    elif action[0] == "approve":
        _, case_id, decision = action
        controller.approve_case(case_id, decision)
    elif action[0] == "reject":
        _, case_id, decision = action
        controller.reject_case(case_id, decision)
    elif action[0] == "escalate":
        _, case_id, decision = action
        controller.escalate_case(case_id, decision)
        
print()
print("Финальное состояние:")
controller.show_cases()
        
        
    
 
            
    
        
        
        
        
        
        
        
        
        

SUCCESS: Case DR-100 created
SUCCESS: Case DR-101 created
SLA Rebate Cases:
DR-100 | SHP-9901 | FastBox | 2 | 5 | 3 | 4200.0 | 300.0 | 378.0 | 0.0 | 0.0 | new | None | False | None
DR-101 | SHP-9902 | QuickShip | 3 | 3 | 0 | 1800.0 | 220.0 | 0.0 | 0.0 | 0.0 | new | None | False | None

Summary:
  total_requested_rebate: 378.0
  total_approved_rebate: 0.0
  total_courier_penalty: 0.0
  delayed_shipments: 1
  contacted_cases: 0
  new: 2
  investigating: 0
  customer_contacted: 0
  ready_for_approval: 0
  approved: 0
  rejected: 0
  escalated: 0
ERROR: Coordinator is required
SUCCESS: Coordinator assigned to DR-100
SUCCESS: Investigation started for DR-100
SUCCESS: Customer contacted for DR-100
SUCCESS: Approved rebate set for DR-100: 180.0
SUCCESS: Case DR-100 marked ready for approval
SUCCESS: Case DR-100 approved
SUCCESS: Case DR-102 created
SUCCESS: Coordinator assigned to DR-102
SUCCESS: Investigation started for DR-102
SUCCESS: Approved rebate set for DR-102: 150.0
SUCCESS: Case DR-